# Main Case: Replicating Lee & Wooldridge (2026), Section 7.2

ECO 4400 &middot; Applied Research & Case Analytics &middot; Journal Replication, Stage 2

The warm-up simplified castle-doctrine adoption to a clean 2x2, comparing one treated cohort against never-adopters over a single pre and post year. Real adoption wasn't nearly that clean. Twenty-one states adopted castle-doctrine laws, in five different years between 2005 and 2009.

1. **The Bridge**. What happens if you run the warm-up's regression on the full, staggered-timing panel, without any special correction?
2. **The Lee & Wooldridge fix**. Their cohort-weighted small-N method (Section 7.2 of the paper) handles staggered timing properly, even with few treated or control units.
3. **Eyeball comparison**. How your numbers stack up against Cheng & Hoekstra's (2013) own published results.

**Required reading:** Cheng & Hoekstra (2013), *"Does Strengthening Self-Defense Law Deter Crime or Escalate Violence?"* (the original castle-doctrine paper) and Lee & Wooldridge (2026), *"Simple Approaches to Inference with Difference-in-Differences Estimators with Small Cross-Sectional Sample Sizes,"* Section 7.2.

**This notebook is a scaffold, not a finished script.** Each section tells you what to compute; you write the code yourself in the empty cell underneath. Work top to bottom -- later cells reuse variables you build in earlier ones, so match the variable names the instructions ask for.

## Setup

Same source as the warm-up: `castle.dta`, loaded directly from a public URL. This time the panel stays whole, covering every adoption cohort across the full 2000-2010 window instead of just 2006 and just two years.

In [ ]:
# Your code here
# - import numpy as np, pandas as pd, statsmodels.api as sm,
#   statsmodels.formula.api as smf, and scipy.stats
# - load castle.dta into a DataFrame called df (same URL as the warm-up)
# - look at df[["sid", "year", "effyear", "cdl", "l_homicide", "popwt"]].head()


## Part 1: The Bridge

`castle.dta` already ships with `cdl`, Cheng & Hoekstra's own treatment variable. It measures the proportion of year `t` that state `i` had an effective Castle Doctrine law in place: zero before adoption, one for a full year under the law, a fraction during the adoption year itself. That variable lets us run the same twoway-fixed-effects regression as the warm-up's Method 4, this time across the whole panel.

In [ ]:
# Your code here
# - regress l_homicide on cdl, with state and year fixed effects
#   (C(sid) + C(year)), clustered by state; save as bridge_unw
# - refit the same regression weighted by popwt (smf.wls instead of
#   smf.ols); save as bridge_w
# - print the cdl coefficient and standard error for both


### Why doesn't this fully solve the staggered-timing problem?

The regression above still runs and gives a sensible-looking number (close to the warm-up's 0.108) -- so what's the issue? With staggered adoption, a standard twoway-fixed-effects regression implicitly uses *already-treated* states as part of the comparison group for *later*-treated states in some of its underlying 2x2 comparisons. Goodman-Bacon (2021) shows these "forbidden comparisons" can bias the pooled coefficient, even though nothing in the regression looks wrong on its face. `castle.dta` is one of the standard textbook examples used to illustrate exactly this problem.

This is the motivation for Lee & Wooldridge's approach in Part 2: instead of pooling every cohort into one regression, transform the panel first so that each unit contributes exactly one clean, appropriately-defined comparison.

## Part 2: The Lee & Wooldridge Small-N Approach

The paper's fix is to collapse each state's entire multi-year history into a single number: how much did this state's homicide rate change, relative to its own pre-treatment self? Once every state is one number, we have a simple cross-section instead of a panel, and an ordinary regression on it sidesteps the staggered-timing and small-cluster-count problems at once.

The paper calls each state's first-treated year **g**. States that adopt in the same year form a **cohort**. A **treated** state uses its own g as the dividing line (equation 7.16). **Control** states never adopted, so they have no natural g of their own; instead, we compute what each one's pre/post change would have looked like under every cohort's g, and average those results, weighted by cohort size (equations 7.11-7.12 and 7.18).

In [ ]:
# Your code here
# - build a "first_treat" column: effyear where it exists, np.inf where
#   it's missing (never-treated states)
# - print how many states fall into each first_treat value (cohort sizes)


In [ ]:
# Your code here
# - build "cohorts": the sorted list of treated states' first_treat years
# - build "cohort_size": a dictionary mapping each cohort year to how many
#   states belong to it
# - build "n_treat": the total number of ever-treated states
# - build "omega": a dictionary mapping each cohort year g to its weight,
#   omega_g = cohort_size[g] / n_treat (equation 7.12)


In [ ]:
# Your code here
# - loop over states (df.groupby("state"))
# - for a treated state: delta_y = mean l_homicide from g_i onward, minus
#   mean l_homicide before g_i (equation 7.16)
# - for a control state (first_treat == np.inf): delta_y is the
#   omega-weighted blend of that same calculation done at EVERY cohort's g
#   (equation 7.18)
# - collect one row per state (state, delta_y, treated) in a list, then
#   build a DataFrame called "transformed" from that list


In [ ]:
# Your code here
# - regress transformed's delta_y on treated using sm.OLS (remember
#   sm.add_constant() first) -- fit once with default (classical) standard
#   errors and once with cov_type="HC3"
# - save the treated coefficient as "tau" (this exact name is used by the
#   comparison cell below)
# - print tau, both standard errors, and the t-statistics
# - compare to the paper: tau_omega = 0.092, SE = 0.057 (t = 1.61), HC3 t = 1.50


## Part 3: Eyeball Comparison with Cheng & Hoekstra (2013)

None of these numbers should match exactly -- they're different specifications on different slices of the same idea. What matters is whether they agree on sign and land in the same neighborhood.

In [ ]:
print("Estimate                                                      Value")
print("-" * 70)
print("Warm-up: single-cohort 2x2 (2006 vs. never-treated, 05->06)   0.108")
print("Bridge: full-panel TWFE, unweighted (this notebook)           0.088")
print("Bridge: full-panel TWFE, population-weighted (this notebook)  0.080")
print(f"Lee & Wooldridge: cohort-weighted small-N (this notebook)      {tau:.3f}")
print("Cheng & Hoekstra Table 5, published range across columns:")
print("  Weighted OLS:    0.080 to 0.100")
print("  Unweighted OLS:  0.058 to 0.088")
print("  Preferred spec (Col 3, region x year FE + controls, weighted): 0.0937")
print("-" * 70)

### A note on the two different "weights" in this exercise

It's easy to conflate these, since they're unrelated ideas that happen to share a name:

- **Cohort weights (`omega_g`)**, used in Part 2, are Lee & Wooldridge's actual method: how to blend a control state's pre/post comparison across every possible treatment-year split. This is the thing being replicated.
- **Population weights (`popwt`)**, used in Part 1's bridge regression and throughout Cheng & Hoekstra's own Table 5, are a standard econometric choice: give larger states more influence in the regression. It has nothing to do with staggered timing or cohorts.

Lee & Wooldridge's method in Part 2 does not use population weights at all.

## Your answer

In 4&ndash;5 sentences: how does your Bridge estimate compare to the warm-up's simplified 2x2? Does your Lee & Wooldridge estimate land closer to Cheng & Hoekstra's published numbers, and why might that be? What does the gap between the Bridge and the Lee & Wooldridge estimate tell you about the risk of running a standard regression on staggered-timing data without correcting for it?

*Double-click this cell to write your answer here.*